# Tema Programare: Principal Component Analysis (PCA) - Ghid Complet

Bun venit la tema de programare despre Principal Component Analysis (PCA)!

Principal Component Analysis este o tehnica puternica de reducere a dimensionalitatii care transforma date cu dimensionalitate mare intr-un spatiu cu dimensionalitate mai mica, pastrand cat mai multa varianta posibil. PCA este folosit pe scara larga pentru vizualizarea datelor, reducerea zgomotului, extragerea de trasaturi si preprocesare inainte de aplicarea algoritmilor de machine learning.

**Ce vei face in aceasta tema:**

* **Implementarea PCA** - Aplici PCA pe date cu dimensionalitate mare, vizualizezi varianta explicata, alegi componentele optime
* **PCA pentru vizualizare** - Reduci seturi de date la 2D/3D pentru vizualizare, colorezi dupa tinta, interpretezi clustere
* **PCA vs Selectia de trasaturi** - Compari reducerea dimensionalitatii cu selectia de trasaturi in ceea ce priveste performanta modelului
* **Pipeline de reducere a dimensionalitatii** - Construiesti un flux complet de lucru: scalezi datele, reduci dimensiunile, integrezi cu modelarea
* **Implementare de la zero** - Implementezi PCA folosind matricea de covarianta si descompunerea in valori proprii
* **Implementarea SVD** - Implementezi Singular Value Decomposition pentru reducerea dimensionalitatii

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul. **Nu adauga si nu modifica cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, asa ca nu te baza pe celulele nou create pentru codul solutiei. Foloseste locurile oferite in notebook.

* Evita folosirea variabilelor globale daca nu este absolut necesar. Evaluatorul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Din acest motiv, variabilele globale pot sa nu fie disponibile la evaluare. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai apasand pe iconita 💾 din coltul stanga sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta sus.
---


## Cuprins
- [Importuri](#imports)
- [1 - Implementarea si analiza PCA](#1)
    - [1.1 - Incarca date cu dimensionalitate mare](#1-1)
    - [1.2 - Aplica PCA](#1-2)
        - **[Exercitiul 1 - apply_pca](#ex-1)**
    - [1.3 - Analizeaza varianta explicata](#1-3)
        - **[Exercitiul 2 - plot_explained_variance](#ex-2)**
    - [1.4 - Alege componentele optime](#1-4)
        - **[Exercitiul 3 - select_n_components](#ex-3)**
- [2 - PCA pentru vizualizare](#2)
    - [2.1 - Redu la 2D](#2-1)
        - **[Exercitiul 4 - visualize_pca_2d](#ex-4)**
    - [2.2 - Redu la 3D](#2-2)
        - **[Exercitiul 5 - visualize_pca_3d](#ex-5)**
    - [2.3 - Interpreteaza incarcaturile componentelor](#2-3)
- [3 - PCA vs Selectia de trasaturi](#3)
    - [3.1 - Metode de selectie a trasaturilor](#3-1)
        - **[Exercitiul 6 - select_k_best_features](#ex-6)**
    - [3.2 - Compara performanta modelului](#3-2)
        - **[Exercitiul 7 - compare_dimensionality_reduction](#ex-7)**
- [4 - Pipeline de reducere a dimensionalitatii](#4)
    - [4.1 - Construieste pipeline-ul complet](#4-1)
        - **[Exercitiul 8 - create_pca_pipeline](#ex-8)**
    - [4.2 - Cross-validare cu pipeline](#4-2)
- [5 - Implementare de la zero](#5)
    - [5.1 - PCA folosind matricea de covarianta](#5-1)
        - **[Exercitiul 9 - pca_from_scratch](#ex-9)**
    - [5.2 - PCA bazat pe SVD](#5-2)
        - **[Exercitiul 10 - pca_using_svd](#ex-10)**
    - [5.3 - Compara implementarile](#5-3)


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.datasets import make_classification, load_digits
from sklearn.metrics import accuracy_score, classification_report

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Import helper functions (if available)
# import helper_utils
# import unittests

<a name='1'></a>
## 1 - Implementarea si analiza PCA

PCA gaseste directiile (componentele principale) din date care maximizeaza varianta. Prima componenta principala capteaza cea mai mare varianta, a doua o capteaza pe a doua ca marime, si asa mai departe.

Matematic, PCA gaseste autovectorii matricei de covarianta:
$$C = \frac{1}{m}X^TX$$

unde $X$ este matricea de date centrata.


<a name='1-1'></a>
### 1.1 - Incarca date cu dimensionalitate mare

Vom lucra cu un set de date care are multe trasaturi pentru a vedea cum ajuta PCA la reducerea dimensionalitatii.


In [ ]:
# Create a synthetic high-dimensional dataset
X_synthetic, y_synthetic = make_classification(
    n_samples=500,
    n_features=50,
    n_informative=10,
    n_redundant=35,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

print(f"Dataset shape: {X_synthetic.shape}")
print(f"Number of features: {X_synthetic.shape[1]}")
print(f"Number of samples: {X_synthetic.shape[0]}")
print(f"Number of classes: {len(np.unique(y_synthetic))}")
print(f"\nClass distribution: {np.bincount(y_synthetic)}")

In [ ]:
# Also load the digits dataset for visualization exercises
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"\nDigits dataset:")
print(f"  Shape: {X_digits.shape}")
print(f"  Features: {X_digits.shape[1]} (8x8 pixel images)")
print(f"  Classes: {len(np.unique(y_digits))} (digits 0-9)")

In [ ]:
# Standardize the data (critical for PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_synthetic)
X_digits_scaled = scaler.fit_transform(X_digits)

print("Data standardized successfully!")
print(f"Mean (should be ~0): {X_scaled.mean(axis=0)[:5]}")
print(f"Std (should be ~1): {X_scaled.std(axis=0)[:5]}")

<a name='1-2'></a>
### 1.2 - Aplica PCA

<a name='ex-1'></a>
**Exercitiul 1 - apply_pca**

Implementeaza o functie pentru a aplica PCA pe date.

**Instructiuni:**
- Foloseste `sklearn.decomposition.PCA`
- Seteaza parametrul `n_components`
- Fa fit si transforma datele
- Returneaza obiectul PCA si datele transformate


In [ ]:
def apply_pca(X, n_components):
    """
    Apply PCA to reduce dimensionality.
    
    Args:
        X: Feature array of shape (m, n)
        n_components: Number of principal components to keep
    
    Returns:
        pca: Fitted PCA object
        X_pca: Transformed data of shape (m, n_components)
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 3 lines)
    pca = None
    X_pca = None
    ### SFÂRȘIT DE COD AICI ###
    
    return pca, X_pca

In [ ]:
# Test your implementation
n_components = 10
pca, X_pca = apply_pca(X_scaled, n_components)

print(f"Original shape: {X_scaled.shape}")
print(f"Reduced shape: {X_pca.shape}")
print(f"\nExplained variance ratio (first {n_components} components):")
for i, var_ratio in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var_ratio:.4f} ({var_ratio*100:.2f}%)")
print(f"\nTotal explained variance: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")

# Unit test (if available)
# unittests.test_apply_pca(apply_pca)

<a name='1-3'></a>
### 1.3 - Analizeaza varianta explicata

<a name='ex-2'></a>
**Exercitiul 2 - plot_explained_variance**

Creeaza vizualizari pentru a intelege cat de multa varianta este explicata de fiecare componenta.

**Instructiuni:**
- Ploteaza varianta explicata individual pentru fiecare componenta
- Ploteaza varianta explicata cumulata
- Adauga o linie orizontala la un prag (de exemplu, 0.95)


In [ ]:
def plot_explained_variance(pca, threshold=0.95):
    """
    Plot explained variance for PCA components.
    
    Args:
        pca: Fitted PCA object
        threshold: Threshold for cumulative variance (for reference line)
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 20 lines)
    explained_variance = None  # Get from pca
    cumulative_variance = None  # Compute cumulative sum
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot 1: Individual explained variance
    # Use ax1.bar or ax1.plot
    
    # Plot 2: Cumulative explained variance
    # Use ax2.plot
    # Add horizontal line at threshold
    
    ### SFÂRȘIT DE COD AICI ###
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Test with full PCA (all components)
pca_full, X_pca_full = apply_pca(X_scaled, X_scaled.shape[1])
plot_explained_variance(pca_full, threshold=0.95)

# Unit test (if available)
# unittests.test_plot_explained_variance(plot_explained_variance)

<a name='1-4'></a>
### 1.4 - Alege componentele optime

<a name='ex-3'></a>
**Exercitiul 3 - select_n_components**

Implementeaza o functie care selecteaza automat numarul de componente necesar pentru a explica un anumit procent din varianta.

**Instructiuni:**
- Calculeaza varianta explicata cumulata
- Gaseste indexul la care varianta cumulata depaseste pragul
- Returneaza numarul optim de componente


In [ ]:
def select_n_components(pca, variance_threshold=0.95):
    """
    Select number of components to retain given variance threshold.
    
    Args:
        pca: Fitted PCA object with all components
        variance_threshold: Minimum cumulative variance to retain
    
    Returns:
        n_components: Number of components needed
        actual_variance: Actual cumulative variance retained
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    cumulative_variance = None
    n_components = None  # Find first index where cumulative > threshold
    actual_variance = None
    ### SFÂRȘIT DE COD AICI ###
    
    return n_components, actual_variance

In [ ]:
# Test your implementation
for threshold in [0.80, 0.90, 0.95, 0.99]:
    n_comp, actual_var = select_n_components(pca_full, threshold)
    print(f"Threshold {threshold:.0%}: Need {n_comp} components (actual variance: {actual_var:.4f})")

# Unit test (if available)
# unittests.test_select_n_components(select_n_components)

<a name='2'></a>
## 2 - PCA pentru vizualizare

Una dintre cele mai puternice aplicatii ale PCA este vizualizarea datelor cu dimensionalitate mare in 2D sau 3D.


<a name='2-1'></a>
### 2.1 - Redu la 2D

<a name='ex-4'></a>
**Exercitiul 4 - visualize_pca_2d**

Creeaza un scatter plot 2D al datelor dupa reducerea PCA, colorat dupa etichetele tintei.

**Instructiuni:**
- Aplica PCA cu 2 componente
- Creeaza un scatter plot cu PC1 pe axa x si PC2 pe axa y
- Coloreaza punctele dupa etichetele claselor
- Adauga varianta explicata in etichetele axelor


In [ ]:
def visualize_pca_2d(X, y, title="PCA 2D Visualization"):
    """
    Visualize data in 2D using PCA.
    
    Args:
        X: Feature array
        y: Target labels
        title: Plot title
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 10 lines)
    # Apply PCA with 2 components
    pca, X_pca = None
    
    # Create scatter plot
    plt.figure(figsize=(10, 8))
    # Use plt.scatter with c=y for coloring
    # Add colorbar
    # Add labels with explained variance
    
    ### SFÂRȘIT DE COD AICI ###
    
    plt.title(title, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Test with synthetic data
visualize_pca_2d(X_scaled, y_synthetic, "PCA 2D: Synthetic Data")

# Test with digits data
visualize_pca_2d(X_digits_scaled, y_digits, "PCA 2D: Handwritten Digits (0-9)")

# Unit test (if available)
# unittests.test_visualize_pca_2d(visualize_pca_2d)

<a name='2-2'></a>
### 2.2 - Redu la 3D

<a name='ex-5'></a>
**Exercitiul 5 - visualize_pca_3d**

Creeaza un scatter plot 3D interactiv folosind primele 3 componente principale.

**Instructiuni:**
- Aplica PCA cu 3 componente
- Foloseste `Axes3D` pentru plotarea 3D
- Coloreaza punctele dupa etichetele claselor
- Eticheteaza axele cu varianta explicata


In [ ]:
def visualize_pca_3d(X, y, title="PCA 3D Visualization"):
    """
    Visualize data in 3D using PCA.
    
    Args:
        X: Feature array
        y: Target labels
        title: Plot title
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 12 lines)
    # Apply PCA with 3 components
    pca, X_pca = None
    
    # Create 3D plot
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create 3D scatter plot
    # Use ax.scatter with c=y for coloring
    # Add labels with explained variance
    
    ### SFÂRȘIT DE COD AICI ###
    
    ax.set_title(title, fontsize=14)
    plt.show()

In [ ]:
# Test with digits data
visualize_pca_3d(X_digits_scaled, y_digits, "PCA 3D: Handwritten Digits (0-9)")

# Unit test (if available)
# unittests.test_visualize_pca_3d(visualize_pca_3d)

<a name='2-3'></a>
### 2.3 - Interpreteaza incarcaturile componentelor

Incarcaturile componentelor arata cat de mult contribuie fiecare trasatura originala la fiecare componenta principala.


In [ ]:
# Analyze component loadings
pca_2d, _ = apply_pca(X_scaled, 2)
loadings = pca_2d.components_.T * np.sqrt(pca_2d.explained_variance_)

# Plot loadings for first two components
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for i in range(2):
    axes[i].bar(range(loadings.shape[0]), loadings[:, i])
    axes[i].set_xlabel('Feature Index', fontsize=12)
    axes[i].set_ylabel('Loading', fontsize=12)
    axes[i].set_title(f'PC{i+1} Loadings (Explained Variance: {pca_2d.explained_variance_ratio_[i]:.2%})', fontsize=14)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 5 features contributing to PC1:")
top_features_pc1 = np.argsort(np.abs(loadings[:, 0]))[-5:][::-1]
for idx in top_features_pc1:
    print(f"  Feature {idx}: {loadings[idx, 0]:.4f}")

<a name='3'></a>
## 3 - PCA vs Selectia de trasaturi

PCA creeaza trasaturi noi (combinatii liniare), in timp ce selectia de trasaturi alege un subset din trasaturile originale. Sa comparam impactul lor asupra performantei modelului.


<a name='3-1'></a>
### 3.1 - Metode de selectie a trasaturilor

<a name='ex-6'></a>
**Exercitiul 6 - select_k_best_features**

Implementeaza selectia de trasaturi folosind teste statistice.

**Instructiuni:**
- Foloseste `sklearn.feature_selection.SelectKBest`
- Foloseste `f_classif` ca functie de scor
- Fa fit si transforma datele


In [ ]:
def select_k_best_features(X, y, k):
    """
    Select k best features using ANOVA F-statistic.
    
    Args:
        X: Feature array
        y: Target array
        k: Number of features to select
    
    Returns:
        selector: Fitted SelectKBest object
        X_selected: Transformed data with k features
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 3 lines)
    selector = None
    X_selected = None
    ### SFÂRȘIT DE COD AICI ###
    
    return selector, X_selected

In [ ]:
# Test your implementation
k = 10
selector, X_selected = select_k_best_features(X_scaled, y_synthetic, k)

print(f"Original features: {X_scaled.shape[1]}")
print(f"Selected features: {X_selected.shape[1]}")
print(f"\nTop {k} feature indices:")
selected_indices = selector.get_support(indices=True)
print(selected_indices)
print(f"\nFeature scores:")
for idx in selected_indices[:5]:
    print(f"  Feature {idx}: {selector.scores_[idx]:.2f}")

# Unit test (if available)
# unittests.test_select_k_best_features(select_k_best_features)

<a name='3-2'></a>
### 3.2 - Compara performanta modelului

<a name='ex-7'></a>
**Exercitiul 7 - compare_dimensionality_reduction**

Compara performanta clasificarii folosind:
1. Trasaturile originale
2. Trasaturi reduse cu PCA
3. Trasaturi selectate

**Instructiuni:**
- Imparte datele in train/test
- Antreneaza regresie logistica pe fiecare varianta
- Compara acuratetile
- Returneaza rezultatele ca dictionar


In [ ]:
def compare_dimensionality_reduction(X, y, n_components=10):
    """
    Compare classification performance with different dimensionality reduction methods.
    
    Args:
        X: Feature array
        y: Target array
        n_components: Number of components/features to use
    
    Returns:
        results: Dictionary with accuracies for each method
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 20 lines)
    # Split data
    X_train, X_test, y_train, y_test = None
    
    results = {}
    
    # 1. Original features
    # Train LogisticRegression and compute accuracy
    
    # 2. PCA-reduced features
    # Apply PCA to train and test
    # Train LogisticRegression and compute accuracy
    
    # 3. Selected features
    # Select features from train and test
    # Train LogisticRegression and compute accuracy
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Test your implementation
results = compare_dimensionality_reduction(X_scaled, y_synthetic, n_components=10)

print("Classification Accuracy Comparison:")
for method, accuracy in results.items():
    print(f"  {method}: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Visualize comparison
plt.figure(figsize=(10, 6))
methods = list(results.keys())
accuracies = list(results.values())
bars = plt.bar(methods, accuracies, color=['blue', 'green', 'orange'], alpha=0.7)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Dimensionality Reduction Methods Comparison', fontsize=14)
plt.ylim([0, 1.0])
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}', ha='center', va='bottom', fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

# Unit test (if available)
# unittests.test_compare_dimensionality_reduction(compare_dimensionality_reduction)

<a name='4'></a>
## 4 - Pipeline de reducere a dimensionalitatii

In practica, ar trebui sa integram reducerea dimensionalitatii intr-un pipeline ML complet.


<a name='4-1'></a>
### 4.1 - Construieste pipeline-ul complet

<a name='ex-8'></a>
**Exercitiul 8 - create_pca_pipeline**

Creeaza un pipeline scikit-learn care:
1. Standardizeaza datele
2. Aplica PCA
3. Antreneaza un clasificator

**Instructiuni:**
- Foloseste `sklearn.pipeline.Pipeline`
- Adauga pasi pentru scaler, PCA si clasificator
- Returneaza obiectul pipeline


In [ ]:
def create_pca_pipeline(n_components=10):
    """
    Create a pipeline with scaling, PCA, and classification.
    
    Args:
        n_components: Number of PCA components
    
    Returns:
        pipeline: sklearn Pipeline object
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    pipeline = None  # Use Pipeline with list of tuples (name, transformer)
    ### SFÂRȘIT DE COD AICI ###
    
    return pipeline

In [ ]:
# Test your implementation
X_train, X_test, y_train, y_test = train_test_split(
    X_synthetic, y_synthetic, test_size=0.3, random_state=42
)

pipeline = create_pca_pipeline(n_components=15)
pipeline.fit(X_train, y_train)

# Evaluate
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print(f"Pipeline with PCA (15 components):")
print(f"  Training accuracy: {train_score:.4f}")
print(f"  Test accuracy: {test_score:.4f}")

# Access PCA object from pipeline
pca_from_pipeline = pipeline.named_steps['pca']
print(f"\nExplained variance: {pca_from_pipeline.explained_variance_ratio_.sum():.4f}")

# Unit test (if available)
# unittests.test_create_pca_pipeline(create_pca_pipeline)

<a name='4-2'></a>
### 4.2 - Cross-validare cu pipeline

Foloseste cross-validare pentru a gasi numarul optim de componente.


In [ ]:
# Test different numbers of components
component_range = [5, 10, 15, 20, 25, 30]
cv_scores = []

print("Cross-validation results:")
for n_comp in component_range:
    pipeline = create_pca_pipeline(n_components=n_comp)
    scores = cross_val_score(pipeline, X_synthetic, y_synthetic, cv=5, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()
    cv_scores.append(mean_score)
    print(f"  n_components={n_comp:2d}: {mean_score:.4f} (+/- {std_score:.4f})")

# Find optimal
optimal_n = component_range[np.argmax(cv_scores)]
print(f"\nOptimal number of components: {optimal_n}")

In [ ]:
# Plot cross-validation results
plt.figure(figsize=(10, 6))
plt.plot(component_range, cv_scores, marker='o', linewidth=2, markersize=8)
plt.axvline(x=optimal_n, color='red', linestyle='--', label=f'Optimal: {optimal_n} components')
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Finding Optimal Number of PCA Components', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

<a name='5'></a>
## 5 - Implementare de la zero

Sa implementam PCA de la zero pentru a intelege matematica!


<a name='5-1'></a>
### 5.1 - PCA folosind matricea de covarianta

<a name='ex-9'></a>
**Exercitiul 9 - pca_from_scratch**

Implementeaza PCA folosind descompunerea in autovalori si autovectori a matricei de covarianta.

**Algoritm:**
1. Centreaza datele: $X_{centered} = X - \bar{X}$
2. Calculeaza matricea de covarianta: $C = \frac{1}{m}X_{centered}^T X_{centered}$
3. Calculeaza autovalorile si autovectorii: $C = V\Lambda V^T$
4. Sorteaza autovectorii dupa autovalori (descrescator)
5. Selecteaza primii k autovectori drept componente principale
6. Proiecteaza datele: $X_{pca} = X_{centered} \cdot W_k$

**Instructiuni:**
- Foloseste `np.cov()` pentru a calcula matricea de covarianta
- Foloseste `np.linalg.eig()` pentru descompunerea in autovalori si autovectori
- Sorteaza autovectorii dupa autovalori


In [ ]:
def pca_from_scratch(X, n_components):
    """
    Implement PCA using eigendecomposition of covariance matrix.
    
    Args:
        X: Feature array of shape (m, n)
        n_components: Number of principal components
    
    Returns:
        X_pca: Transformed data of shape (m, n_components)
        components: Principal components (eigenvectors)
        explained_variance_ratio: Proportion of variance explained
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 15 lines)
    # Step 1: Center the data
    X_mean = None
    X_centered = None
    
    # Step 2: Compute covariance matrix
    cov_matrix = None  # Use np.cov with rowvar=False
    
    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = None  # Use np.linalg.eig
    
    # Step 4: Sort by eigenvalues (descending)
    idx = None  # Use np.argsort
    eigenvalues = None
    eigenvectors = None
    
    # Step 5: Select top k components
    components = None
    
    # Step 6: Project data
    X_pca = None
    
    # Compute explained variance ratio
    explained_variance_ratio = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return X_pca, components, explained_variance_ratio

In [ ]:
# Test your implementation
n_comp = 5
X_pca_scratch, components_scratch, var_ratio_scratch = pca_from_scratch(X_scaled, n_comp)

print(f"From-Scratch PCA Results:")
print(f"  Transformed shape: {X_pca_scratch.shape}")
print(f"  Components shape: {components_scratch.shape}")
print(f"\nExplained variance ratios:")
for i, var in enumerate(var_ratio_scratch, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\nTotal explained variance: {var_ratio_scratch.sum():.4f}")

# Unit test (if available)
# unittests.test_pca_from_scratch(pca_from_scratch)

<a name='5-2'></a>
### 5.2 - PCA bazat pe SVD

<a name='ex-10'></a>
**Exercitiul 10 - pca_using_svd**

PCA poate fi implementat si folosind Singular Value Decomposition (SVD), care este mai stabil numeric.

**Algoritm:**
1. Centreaza datele
2. Calculeaza SVD: $X_{centered} = U\Sigma V^T$
3. Componentele principale sunt coloanele lui $V$
4. Datele transformate: $X_{pca} = U\Sigma$ (sau $X_{centered} \cdot V$)

**Instructiuni:**
- Foloseste `np.linalg.svd()` cu `full_matrices=False`
- Selecteaza primii k vectori singulari
- Calculeaza varianta explicata din valorile singulare


In [ ]:
def pca_using_svd(X, n_components):
    """
    Implement PCA using Singular Value Decomposition.
    
    Args:
        X: Feature array of shape (m, n)
        n_components: Number of principal components
    
    Returns:
        X_pca: Transformed data of shape (m, n_components)
        components: Principal components
        explained_variance_ratio: Proportion of variance explained
    """
    ### ÎNCEPUT DE COD AICI ### (≈ 12 lines)
    m = X.shape[0]
    
    # Step 1: Center the data
    X_mean = None
    X_centered = None
    
    # Step 2: Compute SVD
    U, S, Vt = None  # Use np.linalg.svd with full_matrices=False
    
    # Step 3: Select top k components
    components = None  # V is Vt.T
    
    # Step 4: Project data
    X_pca = None  # Can use U[:, :n_components] * S[:n_components] or X_centered @ components
    
    # Compute explained variance
    explained_variance = None  # Variance = (S**2) / (m - 1)
    total_variance = None
    explained_variance_ratio = None
    
    ### SFÂRȘIT DE COD AICI ###
    
    return X_pca, components, explained_variance_ratio

In [ ]:
# Test your implementation
X_pca_svd, components_svd, var_ratio_svd = pca_using_svd(X_scaled, n_comp)

print(f"SVD-Based PCA Results:")
print(f"  Transformed shape: {X_pca_svd.shape}")
print(f"  Components shape: {components_svd.shape}")
print(f"\nExplained variance ratios:")
for i, var in enumerate(var_ratio_svd, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\nTotal explained variance: {var_ratio_svd.sum():.4f}")

# Unit test (if available)
# unittests.test_pca_using_svd(pca_using_svd)

<a name='5-3'></a>
### 5.3 - Compara implementarile

Sa verificam ca toate cele trei implementari (sklearn, covarianta, SVD) dau rezultate similare.


In [ ]:
# Compare all three implementations
n_comp = 5

# Sklearn
pca_sklearn, X_pca_sklearn = apply_pca(X_scaled, n_comp)
var_sklearn = pca_sklearn.explained_variance_ratio_

# From scratch (covariance)
X_pca_cov, _, var_cov = pca_from_scratch(X_scaled, n_comp)

# SVD
X_pca_svd, _, var_svd = pca_using_svd(X_scaled, n_comp)

# Create comparison table
comparison_df = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(n_comp)],
    'Sklearn': var_sklearn,
    'Covariance': var_cov,
    'SVD': var_svd
})

print("Explained Variance Ratio Comparison:")
print(comparison_df.to_string(index=False))
print(f"\nTotal explained variance:")
print(f"  Sklearn:     {var_sklearn.sum():.6f}")
print(f"  Covariance:  {var_cov.sum():.6f}")
print(f"  SVD:         {var_svd.sum():.6f}")

In [ ]:
# Visualize comparison
x = np.arange(n_comp)
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, var_sklearn, width, label='Sklearn', alpha=0.8)
ax.bar(x, var_cov, width, label='Covariance', alpha=0.8)
ax.bar(x + width, var_svd, width, label='SVD', alpha=0.8)

ax.set_xlabel('Principal Component', fontsize=12)
ax.set_ylabel('Explained Variance Ratio', fontsize=12)
ax.set_title('PCA Implementation Comparison', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([f'PC{i+1}' for i in range(n_comp)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.show()

print("\nAll implementations produce nearly identical results!")

In [ ]:
# Compare reconstruction error
def reconstruction_error(X_original, X_reduced, components, mean):
    """Compute reconstruction error."""
    X_reconstructed = X_reduced @ components.T + mean
    return np.mean((X_original - X_reconstructed)**2)

X_mean = X_scaled.mean(axis=0)

# Note: sklearn uses slightly different centering
error_cov = reconstruction_error(X_scaled, X_pca_cov, components_scratch.T, X_mean)
error_svd = reconstruction_error(X_scaled, X_pca_svd, components_svd.T, X_mean)

print("Reconstruction Errors (MSE):")
print(f"  Covariance method: {error_cov:.6f}")
print(f"  SVD method:        {error_svd:.6f}")
print(f"\nBoth methods achieve similar reconstruction quality!")

## Rezumat si pasii urmatori

**Felicitari!** Ai finalizat tema despre PCA. Ai:

- Implementat PCA si ai analizat varianta explicata  
- Ales numarul optim de componente pe baza pragului de varianta  
- Vizualizat date cu dimensionalitate mare in 2D si 3D  
- Interpretat incarcaturile componentelor principale  
- Comparat PCA cu metode de selectie a trasaturilor  
- Construit pipeline-uri complete de reducere a dimensionalitatii  
- Implementat PCA de la zero folosind descompunerea in autovalori a matricei de covarianta  
- Implementat PCA folosind SVD  

**Idei-cheie:**
- PCA creeaza trasaturi necorelate care maximizeaza varianta
- Varianta explicata te ajuta sa alegi numarul potrivit de componente
- PCA este excelent pentru vizualizare si reducerea zgomotului
- Standardizeaza intotdeauna datele inainte de PCA
- SVD este mai stabil numeric decat descompunerea in autovalori
- Pipeline-urile integreaza elegant preprocesarea si modelarea

**Cand sa folosesti PCA:**
- Date cu dimensionalitate mare (multe trasaturi)
- Trasaturi corelate
- Ai nevoie sa vizualizezi datele
- Vrei sa reduci zgomotul
- Vrei sa accelerezi antrenarea cu mai putine trasaturi

**Cand sa NU folosesti PCA:**
- Ai nevoie sa interpretezi trasaturile originale
- Trasaturile sunt deja independente
- Numar mic de trasaturi
- Relatii neliniare (ia in calcul kernel PCA sau autoencodere)

**Pasii urmatori:**
- Incearca PCA pe seturi de date reale (imagini, text, genomica)
- Exploreaza kernel PCA pentru reducere neliniara a dimensionalitatii
- Invata si alte tehnici: t-SNE, UMAP, autoencodere
- Aplica PCA in pipeline-uri de deep learning!
